# main.tex Post-Quench 2DTN Benchmark

Runs the square-lattice TFIM post-quench protocols with the default paper-style 2DTN settings: nearest-neighbor interactions, `chi_2D=40`, `chi_2D'=24`, `r_2D=32`, `dt J=0.01`, and boundary-MPS measurements.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from ryd_gate.protocols.lattice_dynamics import TFIMQuenchProtocol
from tn_common import create_tn_lattice_spec, simulate_tn

In [ ]:
L = 10
HX_VALUES = [0.5, 2.0, 3.0]
T_FINAL = 3.0
T_EVAL_INTERVAL = 0.1
V_NN = 4.0  # J = V_nn / 4, so V_nn=4 gives J=1
INTERACTION_MODE = "nn"
BACKEND = "2dtn"
OUTPUT_DIR = repo_root / "scripts" / "notebooks" / "results" / "quench_benchmark"

BACKEND_OPTIONS = {
    "chi_2d": 40,
    "chi_2d_prime": 24,
    "dt": 0.01,
    "svd_min": 1e-10,
    "measurement_alg": "boundarymps",
    "measurement_bond_dim": 32,
    "julia_cmd": "julia",
    "source_bashrc": True,
    "use_cuda": False,
    "timeout": 7200,
}

t_eval = np.arange(0.0, T_FINAL + 1e-12, T_EVAL_INTERVAL)

In [ ]:
def run_quench(hx):
    spec = create_tn_lattice_spec(
        L,
        L,
        V_nn=V_NN,
        Omega=2.0 * hx,
        level_structure="1r",
        interaction_mode=INTERACTION_MODE,
        ordering="snake",
    )
    protocol = TFIMQuenchProtocol(hx=hx, hz=0.0, t_gate=T_FINAL)
    return simulate_tn(
        spec,
        protocol,
        [],
        initial_state="all_ground",
        backend=BACKEND,
        t_eval=t_eval,
        observables=["sigma_z", "czz_centerline"],
        backend_options=BACKEND_OPTIONS,
    )


def save_result(hx, result):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / f"quench_L{L}_hx{str(hx).replace('.', 'p')}_{BACKEND}.npz"
    arrays = {"times": np.asarray(result.times)}
    for name, value in (result.metadata.get("obs") or {}).items():
        arrays[f"obs_{name}"] = np.asarray(value)
    if "final_sigma_z" in result.metadata:
        arrays["final_sigma_z"] = np.asarray(result.metadata["final_sigma_z"])
    if "truncation_error" in result.metadata:
        arrays["truncation_error"] = np.asarray(result.metadata["truncation_error"])
    metadata = {
        "L": L,
        "hx": hx,
        "t_final": T_FINAL,
        "interaction_mode": INTERACTION_MODE,
        "backend": BACKEND,
        "backend_options": BACKEND_OPTIONS,
        "result_backend": result.metadata.get("backend"),
        "result_method": result.metadata.get("method"),
    }
    arrays["metadata_json"] = np.asarray(json.dumps(metadata, sort_keys=True))
    np.savez(path, **arrays)
    return path

In [ ]:
results = {}
for hx in HX_VALUES:
    result = run_quench(hx)
    path = save_result(hx, result)
    results[hx] = result
    print(f"saved h_x/J={hx}: {path}")

In [ ]:
center_site = (L // 2) * L + (L // 2)
center_col = L // 2
line_cols = [iy for iy in range(L) if iy != center_col]
nearest_corr_idx = min(range(len(line_cols)), key=lambda idx: abs(line_cols[idx] - center_col))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for hx, result in results.items():
    obs = result.metadata["obs"]
    axes[0].plot(result.times, obs["sigma_z"][:, center_site], label=f"h_x/J={hx}")
    axes[1].plot(result.times, obs["czz_centerline"][:, nearest_corr_idx], label=f"h_x/J={hx}")

axes[0].set_xlabel("tJ")
axes[0].set_ylabel(r"$\langle \sigma^z_{r_0} \rangle$")
axes[1].set_xlabel("tJ")
axes[1].set_ylabel(r"$C(\delta_1)$")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
fig.tight_layout()